# Model Results Scorecard

This notebook summarizes the current performance of the core modeling pieces in the real-estate NLP project:

- hybrid imputation holdout tests
- price prediction model comparison
- listing cluster output summary

The main goal is to answer a practical question: are the current results good enough to trust, and where should the next refinement go?

In [ ]:
import os
from pathlib import Path
import json

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

mpl_config_dir = PROJECT_ROOT / '.matplotlib'
mpl_config_dir.mkdir(exist_ok=True)
os.environ['MPLCONFIGDIR'] = str(mpl_config_dir)

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 60)

PROCESSED = PROJECT_ROOT / 'data' / 'processed'
impute_report = json.loads((PROCESSED / 'hybrid_imputer_evaluation_report.json').read_text(encoding='utf-8'))
price_report = json.loads((PROCESSED / 'price_model_comparison_report.json').read_text(encoding='utf-8'))
cluster_report = json.loads((PROCESSED / 'listing_cluster_report.json').read_text(encoding='utf-8'))

impute_df = pd.DataFrame(impute_report['results'])
price_df = pd.DataFrame(price_report['results'])
cluster_df = pd.DataFrame(cluster_report)

impute_df.head()

## Hybrid imputer holdout results

These numbers come from a masking experiment: we hide known values, ask the imputer to recover them, and compare the prediction to the truth.

In [ ]:
impute_df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(data=impute_df, x='tolerance_accuracy', y='target', hue='mode', ax=axes[0])
axes[0].set_title('Imputation Tolerance Accuracy')
axes[0].set_xlabel('Accuracy within target-specific tolerance')
axes[0].set_ylabel('Target')

sns.barplot(data=impute_df, x='regex_coverage', y='target', hue='mode', ax=axes[1])
axes[1].set_title('Regex Coverage by Target')
axes[1].set_xlabel('Share recovered directly from text')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

### Read on the imputer

- `beds` and `baths` are strong. They have high tolerance accuracy and decent regex coverage.
- `stories` is fairly solid too.
- `sqft` is much harder. That is expected because square footage is noisy, large-scale, and not always stated cleanly in text.
- `year_built` is currently weak. The text does not mention it often enough, and the model has a much harder time recovering it reliably.

## Price model comparison

This compares three setups using the price-safe table:

- text-only
- structured-only
- text plus structured

In [ ]:
price_df

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
sns.barplot(data=price_df, x='model', y='mae', palette='crest', ax=axes[0])
axes[0].set_title('MAE')
axes[0].tick_params(axis='x', rotation=20)

sns.barplot(data=price_df, x='model', y='rmse', palette='flare', ax=axes[1])
axes[1].set_title('RMSE')
axes[1].tick_params(axis='x', rotation=20)

sns.barplot(data=price_df, x='model', y='r2', palette='magma', ax=axes[2])
axes[2].set_title('R-squared')
axes[2].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.show()

### Read on the price models

- The text-only model is useful but clearly weaker than the structured model.
- The structured model is currently the strongest performer.
- The tuned combined blend landed on a full structured weight, which tells us the current text signal is not yet adding enough incremental value beyond the structured side.

That is not a bad result. It is actually a useful finding: listing language carries price signal, but the current text pipeline is not yet beating a strong structured baseline.

## Cluster summary

The listing clustering layer is working and already producing segments with noticeably different average prices and quality scores.

In [ ]:
cluster_df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(data=cluster_df, x='row_count', y='cluster_id', palette='viridis', ax=axes[0])
axes[0].set_title('Cluster Sizes')
axes[0].set_xlabel('Rows')
axes[0].set_ylabel('Cluster')

sns.barplot(data=cluster_df, x='avg_price', y='cluster_id', palette='rocket', ax=axes[1])
axes[1].set_title('Average Price by Cluster')
axes[1].set_xlabel('Average price')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

## Bottom line

What looks good right now:
- the hybrid imputer is genuinely useful for `beds`, `baths`, and `stories`
- the price pipeline has a reasonable structured baseline around `R^2 = 0.49`
- the clustering layer is already segmenting the corpus into meaningfully different price bands

What needs more work:
- `sqft` and especially `year_built` imputation
- making text add more value in the price model
- turning clusters into interpretable themes with top terms and example listings

## Recommended next steps

1. tighten the hybrid imputer for `sqft` and `year_built`
2. add topic labels / top terms to the clusters
3. build a small similar-listing demo using the TF-IDF search script
4. improve the combined price model with better text features or embeddings